In [ ]:
!pip install causal-tracer

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from causal_tracer import CausalTracer
from typing import List, Dict, Any

In [ ]:
model_name = "gpt2-large"

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tracer = CausalTracer(model, tokenizer)



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:


NATIONALITY_TO_COUNTRY_MAP = {
    'argentina': 'Argentina',
    'boliviana': 'Bolivia',
    'chilena': 'Chile',
    'colombiana': 'Colombia',
    'ecuatoriana': 'Ecuador',
    'paraguaya': 'Paraguay',
    'peruana': 'Peru',
    'uruguaya': 'Uruguay',
    'venezolana': 'Venezuela',
}


def load_and_transform_nationality_facts(file_path: str) -> pd.DataFrame:

    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame()

    df.columns = ['source', 'relation', 'target']

    filtered_df = df[df['relation'] == 'es de nacionalidad'].copy()

    filtered_df['nationality_clean'] = filtered_df['target'].str.replace(r'[^\w\s]', '', regex=True).str.lower()
    filtered_df['country'] = filtered_df['nationality_clean'].map(NATIONALITY_TO_COUNTRY_MAP)
    mapped_df = filtered_df.dropna(subset=['country'])
    mapped_df = mapped_df.reset_index(drop=True)
    print(f"Loaded {len(df)} total facts. Filtered and mapped {len(mapped_df)} nationality facts.")


    spanish_facts = mapped_df.copy()
    spanish_facts['prompt'] = "El autor " + spanish_facts['source'] + ' es de nacionalidad'
    spanish_facts['subject'] = spanish_facts['source']
    spanish_facts['correct_answer'] = spanish_facts['nationality_clean']
    spanish_facts['language'] = 'es'

    english_facts = mapped_df.copy()
    english_facts['prompt'] = "The author " + english_facts['source'] + ' comes from the country of'
    english_facts['subject'] = english_facts['source']
    english_facts['correct_answer'] = english_facts['country']
    english_facts['language'] = 'en'


    mapped_df['fact_id'] = mapped_df.index.to_series()
    spanish_facts['fact_id'] = spanish_facts.index.to_series()
    english_facts['fact_id'] = english_facts.index.to_series()

    final_df = pd.concat([spanish_facts, english_facts])


    final_df['lang_sort'] = final_df['language'].apply(lambda x: 0 if x == 'es' else 1)

    final_df = final_df.sort_values(by=['fact_id', 'lang_sort']).drop(columns=['fact_id', 'lang_sort'])

    final_df = final_df[['prompt', 'subject', 'correct_answer', 'language']]

    return final_df.reset_index(drop=True)


In [ ]:
def normalize_string(s: str) -> str:
    if not isinstance(s, str):
        return ""

    cleaned_s = s.strip().lower()

    cleaned_s = cleaned_s.rstrip('.,!?"\'')


    if not cleaned_s and s.strip():
        return "_PUNCT_ONLY_"

    return cleaned_s.strip()

def extract_hidden_flow_metrics(
    tracer: CausalTracer,
    prompt: str,
    subject: str,
    correct_answer: str,
    language: str
) -> Dict[str, Any]:


    try:

        flow = tracer.calculate_hidden_flow(
            prompt=prompt,
            subject=subject,
        )
    except Exception as e:
        print(f"Error tracing fact (Prompt: {prompt}): {e}")
        return {
            'prompt': prompt,
            'subject': subject,
            'correct_answer': correct_answer,
            'language': language,
            'is_correct': False,
            'traced_answer': 'ERROR',
            'mean_causal_effect': np.nan,
            'max_causal_effect': np.nan,
            'std_causal_effect': np.nan,
            'peak_layer_index': np.nan,
            'peak_score_at_layer': np.nan
        }

    traced_answer = flow.answer
    is_correct = False

    normalized_expected = normalize_string(correct_answer)
    normalized_traced = normalize_string(traced_answer)


    if normalized_traced == "_PUNCT_ONLY_":
        is_correct = False
    elif normalized_expected == "_PUNCT_ONLY_":
        is_correct = False
    else:
        if normalized_traced.startswith(normalized_expected) or normalized_expected.startswith(normalized_traced):
            is_correct = True

    scores_np = flow.scores.cpu().numpy()
    scores_flattened = scores_np.flatten()
    mean_by_layer = np.mean(scores_np, axis=1)

    mean_causal_effect = np.mean(scores_flattened)
    max_causal_effect = np.max(scores_flattened)
    std_causal_effect = np.std(scores_flattened)

    peak_layer_index = np.argmax(mean_by_layer)
    peak_score = mean_by_layer[peak_layer_index]

    return {
        "region": "SOUTH AMERICA",
        'prompt': prompt,
        'subject': subject,
        'correct_answer': correct_answer,
        'language': language,
        'is_correct': is_correct,
        'traced_answer': traced_answer,
        'mean_causal_effect': mean_causal_effect,
        'max_causal_effect': max_causal_effect,
        'std_causal_effect': std_causal_effect,
        'peak_layer_index': peak_layer_index,
        'peak_score_at_layer': peak_score
    }

In [ ]:

def run_causal_tracing_pipeline_nationality(
    file_path: str,
    tracer: CausalTracer,
    start_index: int = 0,
    end_index: int = None
) -> pd.DataFrame:

    print(f"Starting pipeline. Tracing model: {tracer.model.config._name_or_path}")

    full_facts_df = load_and_transform_nationality_facts(file_path)

    if full_facts_df.empty:
        return pd.DataFrame()

    facts_df = full_facts_df

    total_rows = len(facts_df)
    results_list = []

    stop_at = end_index if end_index is not None and end_index <= total_rows else total_rows

    if start_index >= stop_at:
        print(f"Start index ({start_index}) is at or after the end index ({stop_at}). Returning empty DataFrame.")
        return pd.DataFrame()

    print(f"Processing rows from index {start_index} up to index {stop_at - 1}.")


    for index in range(start_index, stop_at):
        row = facts_df.iloc[index]

        metrics = extract_hidden_flow_metrics(
            tracer,
            prompt=row['prompt'],
            subject=row['subject'],
            correct_answer=row['correct_answer'],
            language=row['language']
        )
        results_list.append(metrics)


    final_results_df = pd.DataFrame(results_list)
    print("\n--- Pipeline Complete ---")

    return final_results_df

In [ ]:
file_to_process = "tripletas.csv"

In [ ]:
results_df = run_causal_tracing_pipeline_nationality(file_to_process, tracer, start_index =1500 , end_index = 1550)

Starting pipeline. Tracing model: gpt2-large
Loaded 24792 total facts. Filtered and mapped 2068 nationality facts.
Processing rows from index 1500 up to index 1549.

--- Pipeline Complete ---


In [ ]:
results_df.to_csv('results_nationality.csv', index=False)

In [ ]:
from google.colab import files
files.download('results_nationality.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from typing import List


def load_and_transform_author_facts(file_path: str) -> pd.DataFrame:

    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame()

    df.columns = ['author', 'relation', 'book']


    mapped_df = df[df['relation'] == 'es autor de la obra'].copy()


    mapped_df = mapped_df.reset_index(drop=True)

    print(f"Loaded {len(df)} total facts. Filtered and mapped {len(mapped_df)} author facts.")

    spanish_facts = mapped_df.copy()
    spanish_facts['prompt'] = "El autor del libro " + spanish_facts['book'] + ' es'
    spanish_facts['subject'] = spanish_facts['book']
    spanish_facts['correct_answer'] = spanish_facts['author']
    spanish_facts['language'] = 'es'

    english_facts = mapped_df.copy()
    english_facts['prompt'] = "The book " + english_facts['book'] + ' was authored by'
    english_facts['subject'] = english_facts['book']
    english_facts['correct_answer'] = english_facts['author']
    english_facts['language'] = 'en'

    mapped_df['fact_id'] = mapped_df.index.to_series()
    spanish_facts['fact_id'] = mapped_df['fact_id']
    english_facts['fact_id'] = mapped_df['fact_id']

    final_df = pd.concat([spanish_facts, english_facts])

    final_df['lang_sort'] = final_df['language'].apply(lambda x: 0 if x == 'es' else 1)

    final_df = final_df.sort_values(by=['fact_id', 'lang_sort']).drop(columns=['fact_id', 'lang_sort'])

    final_df = final_df[['prompt', 'subject', 'correct_answer', 'language']]

    return final_df.reset_index(drop=True)

In [ ]:

def run_causal_tracing_pipeline_authorship(
    file_path: str,
    tracer: CausalTracer,
    start_index: int = 0,
    end_index: int = None
) -> pd.DataFrame:

    print(f"Starting pipeline. Tracing model: {tracer.model.config._name_or_path}")

    full_facts_df = load_and_transform_author_facts(file_path)

    if full_facts_df.empty:
        return pd.DataFrame()

    facts_df = full_facts_df

    total_rows = len(facts_df)
    results_list = []

    stop_at = end_index if end_index is not None and end_index <= total_rows else total_rows

    if start_index >= stop_at:
        print(f"Start index ({start_index}) is at or after the end index ({stop_at}). Returning empty DataFrame.")
        return pd.DataFrame()

    print(f"Processing rows from index {start_index} up to index {stop_at - 1}.")


    for index in range(start_index, stop_at):
        row = facts_df.iloc[index]

        metrics = extract_hidden_flow_metrics(
            tracer,
            prompt=row['prompt'],
            subject=row['subject'],
            correct_answer=row['correct_answer'],
            language=row['language']
        )
        results_list.append(metrics)


    final_results_df = pd.DataFrame(results_list)
    print("\n--- Pipeline Complete ---")

    return final_results_df

In [ ]:
file_to_process = "tripletas.csv"

In [ ]:
results_df = run_causal_tracing_pipeline_authorship(file_to_process, tracer, start_index =1000 , end_index = 1050)

Starting pipeline. Tracing model: gpt2-large
Loaded 24792 total facts. Filtered and mapped 4651 author facts.
Processing rows from index 1000 up to index 1049.

--- Pipeline Complete ---


In [ ]:
results_df.to_csv('results_authorship.csv', index=False)

In [ ]:
from google.colab import files
files.download('results_authorship.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>